# 4-Bit Quantization mistralai/Mistral-Large-Instruct-2407

## Inference with mistralai/Mistral-Large-Instruct-2407

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

pretrained_model_id = "mistralai/Mistral-Large-Instruct-2407"

# Load the model with Float16 precision on a single GPU
model = AutoModelForCausalLM.from_pretrained(pretrained_model_id, torch_dtype=torch.float16, device_map="cuda:0")
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_id)

prompt = "In numerical mathematics, quantization is"
input_ids = tokenizer.encode(prompt, return_tensors='pt')
output = model.generate(input_ids, max_new_tokens=50)
tokenizer.decode(output[0], skip_special_tokens=True)


## Quantizing mistralai/Mistral-Large-Instruct-2407

In [ ]:
import os
# Re-start the python kernel to clear previous allocated resources
os._exit(0)

In [ ]:
import os
from datasets import load_dataset
from gptqmodel import GPTQModel, QuantizeConfig
from transformers import AutoTokenizer
import torch

In [ ]:
os.environ['HF_TOKEN']="<YOUR_HF_TOKEN>"

### Model preparation and calibration dataset

In [ ]:
# The non-quantized model tags:
pretrained_model_id = "mistralai/Mistral-Large-Instruct-2407"

# Load the tokenizer associated to the pretrained model
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_id)

In [ ]:
def get_wikitext2(tokenizer, nsamples, seqlen):
    '''
    Loads the dataset and tokenize it
    '''
    
    # Select text larger than seqlen
    traindata = load_dataset("Salesforce/wikitext", 
                              "wikitext-2-raw-v1", 
                              split="train").filter(lambda x: len(x["text"]) >= seqlen)
    
    tokenized_data = [(example["text"], tokenizer(example["text"])) for example in traindata.select(range(nsamples))]

    return zip(*tokenized_data)

# Explore the first 3 samples
text_example, tokenized_example  = get_wikitext2(tokenizer, nsamples=3, seqlen=5)

for text, tokenized_text in zip(text_example,tokenized_example):
    print(f"EXAMPLE TEXT:\n{text}\nTOKENIZED TEXT:\n{tokenized_text} \n") 

### Load the non-quantized model and set the quantization parameters

In [ ]:
quantize_config = QuantizeConfig(
    bits=4,  # quantize model to 4-bit (INT4).
    group_size=128,  # it is recommended to set the value to 128
)

# Load the original (non-quantized) model.
model = GPTQModel.load(pretrained_model_id, 
                      quantize_config, 
                      trust_remote_code=True # Hugging Face Datasets/load_dataset parameter
                      )

In [ ]:
# Explore the non-quantized model's architecture
model

### Quantizing mistralai/Mistral-Large-Instruct-2407

In [ ]:
# Loading the calibration dataset with 512 samples in the data
_, calibration_dataset = get_wikitext2(tokenizer, nsamples=512, seqlen=1024)

In [ ]:
# Quantize the original model
model.quantize(calibration_dataset, batch_size=32)

# Save the model locally with a new tag for the new quantized version
quantized_model_id = f"quantized_{pretrained_model_id.split('/')[1]}_4bit"
model.save(quantized_model_id)

In [ ]:
# Explore the quantized model's architecture
model

### Performing inference with the quantized model

In [ ]:
pretrained_model_id = "mistralai/Mistral-Large-Instruct-2407"
quantized_model_id = f"quantized_{pretrained_model_id.split('/')[1]}_4bit"
tokenizer = AutoTokenizer.from_pretrained(quantized_model_id, use_fast=True,)

In [ ]:
# Load the quantized model from local and on a single GPU
device = "cuda:0" 
model = GPTQModel.load(quantized_model_id, device=device, trust_remote_code=True)

# Inference using model.generate
prompt = "In numerical mathematics, quantization is"
output = model.generate(**tokenizer(prompt, return_tensors="pt").to(device), max_new_tokens=50)

# Print the output
print(tokenizer.decode(output[0], skip_special_tokens=True))